# 01.3 — Schemas and validation

[Previous: 01.2](01.2-behavior-and-composition.ipynb) · [Class index and setup](README.md) · [Next: 01.4](01.4-serialization-and-json-schema.ipynb)

**Purpose:** Define the shape and allowed values of a record before collecting it.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the class index's 210-minute allowance. The separately assessed project adds workload.

## What you will be able to do

- Distinguish a Python type hint from runtime input validation.
- Define a Pydantic model and interpret field-level errors.
- Compare default conversion with a deliberate strict field.

## Starting point

A Report is a named collection. This file supplies the class again and introduces its validated record type. You need dictionaries and try/except; the examples explain the new library calls.

This file includes all definitions and inputs it needs. Read the parts in order, but start this one with **Restart Kernel → Run All** in VS Code. Do not execute another notebook or an earlier exercise solution to prepare it. Empty answer cells are safe during Run All.

Reference solutions stay in separate cells: attempt the task, uncomment its `%load` line, and run that cell to reveal source. Inspect the result before running the populated cell separately. Keep your answer in its own cell because loading can replace the invoking cell. A loaded solution remains in the notebook even after restarting the kernel.

In [ ]:
from pathlib import Path
import sys

if not Path("solutions/exercise_01_3_e1.py").is_file():
    raise FileNotFoundError(
        "Open 01-oop-and-schemas.code-workspace in VS Code and restart the kernel. "
        "The working directory must be the class folder containing README.md and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class files are available.")

## In class · 60–80 minutes — A hint is not a runtime check

A type annotation tells readers and tools what kind of value is intended. Python does not automatically reject an argument just because it disagrees with that annotation. The following deliberately simple function returns exactly what it receives.

In [ ]:
def echo_quantity(value: float) -> float:
    return value


result = echo_quantity("3.5")
print("Returned:", result, "Actual type:", type(result).__name__)
assert isinstance(result, str)

The returned value is still a string. An editor or static checker can warn about the call, but a hint alone has not validated it. Data arriving from files or external services needs an explicit boundary check.

### A schema is a record contract

`Observation` below inherits Pydantic's validation behavior from `BaseModel`. Its fields describe a nonempty label and a finite, nonnegative quantity. `Field(min_length=1)` and `Field(ge=0)` add constraints; `allow_inf_nan=False` excludes non-finite numeric values. These choices fit our quantity example, not every scientific measurement.

No custom constructor is needed. Pydantic supplies one from the field definitions. Use `model_validate` to check a dictionary.

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Observation(BaseModel):
    """A nonempty label and a finite, nonnegative quantity."""
    label: str = Field(min_length=1)
    value: float = Field(ge=0, allow_inf_nan=False)

In [ ]:
record = Observation.model_validate({"label": "north", "value": 2.5})
converted = Observation.model_validate({"label": "south", "value": "3.5"})
print(record)
print("Converted value:", converted.value, "Type:", type(converted.value).__name__)
assert converted.value == 3.5 and isinstance(converted.value, float)

### Visual guide — Accept, convert, or reject

<img src="images/validation-outcomes.png" alt="Number 2.5 validates; numeric string 2.5 converts under the default policy; word many raises a validation error." width="760" style="max-width: 100%; height: auto; background-color: white;">

The same Observation contract accepts a number, converts a numeric string by default, and rejects invalid text. The later strict-field example changes conversion policy; type hints alone perform none of these checks.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/validation-outcomes.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

The numeric string is accepted and converted under default validation. Validation can normalize data as well as reject it. Whether that conversion is appropriate is a policy decision; the strict-field follow-up below shows an alternative.

### Inspect a failure instead of hiding it

An invalid input raises `ValidationError`. Its `errors()` result identifies the field location, error category, and explanation. Catch that expected error at the input boundary so you can report a useful reason. Do not catch every exception and pretend the program succeeded.

In [ ]:
try:
    Observation.model_validate({"label": "west", "value": -2})
except ValidationError as error:
    for detail in error.errors():
        print("Field:", detail["loc"], "Category:", detail["type"], "Reason:", detail["msg"])

### Compose a report from validated observations

The report now contains Observation objects, so `total` reads `record.value` instead of `record["value"]`. The division of responsibility is explicit: validate the raw dictionary, then add the resulting object. An annotation would document the expected input but would not itself enforce this boundary.

In [ ]:
class Report:
    """Collect observations already validated at the input boundary."""

    def __init__(self, name):
        self.name = name
        self.records = []

    def add(self, observation):
        self.records.append(observation)

    def total(self):
        return sum(observation.value for observation in self.records)

In [ ]:
validated = Observation.model_validate({"label": "north", "value": 2.5})
report = Report("validated daily")
report.add(validated)
print("Validated report total:", report.total())
assert report.total() == 2.5

### 🧩 Exercise 01.3.E1 — Accept, convert, or reject?

> **In class · 10 minutes**
>
> **Goal:** Apply the data contract and explain rejected fields.
>
> **Start here:** The Observation definition above. Use these six raw dictionaries: north/4, south/"3.5", an empty label/1, west/-2, east without value, and hill/"broken".
>
> **Task:** Create those dictionaries, validate each, collect accepted records, and record the index plus field reason for each rejection. Predict the outcomes before running the loop.
>
> **Check:** Two records are accepted and four rejected. The accepted numeric string becomes 3.5. Missing value, empty label, negative value, and nonnumeric text have identifiable reasons.
>
> **Hint:** Catch ValidationError around one record at a time. error.errors() describes the failure; do not stop the whole batch at its first bad row.

In [ ]:
# Your attempt for 01.3.E1.
# Keep your code and explanation here, separate from the reference cell.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_01_3_e1.py

## Guided follow-up — Independent work · 15 minutes

### Make one field strict deliberately

Default conversion can be useful when a file represents all numeric cells as text. If the source is supposed to send actual numbers, silently accepting strings can conceal a source-format change. A strict field lets you choose that boundary policy. This small comparison model repeats the fields so the difference is visible.

In [ ]:
class StrictObservation(BaseModel):
    label: str = Field(min_length=1)
    value: float = Field(ge=0, allow_inf_nan=False, strict=True)


strict_number = StrictObservation.model_validate({"label": "north", "value": 2.5})
print("Actual number accepted:", strict_number.value)
try:
    StrictObservation.model_validate({"label": "south", "value": "3.5"})
except ValidationError as error:
    print("Strict field rejected the string:", error.errors()[0]["loc"])

Strictness is chosen here for the `value` field; it is not a claim that every form of conversion is always wrong. Keep the chosen behavior documented and checked. Do not memorize an error message's exact wording; understand the field and policy that produced it.

### Know what the contract does not say

A minimum string length rejects `""`; it does not automatically reject a string containing only spaces or restrict labels to a known vocabulary. A nonnegative number may still be an incorrect observation. Structural validation is one boundary, not a complete data-quality guarantee.

The report also does not validate every object passed to `add`. In this design the caller performs that operation first. If a future interface must accept raw dictionaries, place validation explicitly in that interface rather than assuming a type annotation performs runtime validation.

### Completion check and reflection

Within this follow-up, explain the three outcomes for an actual number, a numeric string, and invalid text. Identify a missing-field failure and a value-constraint failure. Choose whether default or strict input is appropriate for one source and justify it.

**Optional extension · 15 minutes, excluded from required time:** try a whitespace-only label and explain an additional business rule you would need. Do not add a custom validation framework.

## References

- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/)
- [Strict mode](https://docs.pydantic.dev/latest/concepts/strict_mode/)
- [Previous: behavior and composition](01.2-behavior-and-composition.ipynb)
- [Next: serialization and JSON Schema](01.4-serialization-and-json-schema.ipynb)
- [Class index](README.md)